In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    ArrayType,
    MapType
)

import re


# ============================================================
# 1. READ PARAMETERS
# ============================================================

catalog = dbutils.widgets.get("catalog")
target_schema = dbutils.widgets.get("target_schema")
target_table = dbutils.widgets.get("target_table")

source_file_path = dbutils.widgets.get("source_file_path")
row_tag = dbutils.widgets.get("row_tag")

parse_mode = dbutils.widgets.get("parse_mode")
write_mode = dbutils.widgets.get("write_mode")
table_comment = dbutils.widgets.get("table_comment")

target_table_full_name = (
    f"`{catalog}`.`{target_schema}`.`{target_table}`"
)

print("Source file:", source_file_path)
print("Target table:", target_table_full_name)
print("XML row tag:", row_tag)


# ============================================================
# 2. VALIDATE PARAMETERS
# ============================================================

if not source_file_path:
    raise ValueError("Source file path cannot be empty.")

if not row_tag:
    raise ValueError("XML row tag cannot be empty.")

if write_mode.lower() not in ["overwrite", "append"]:
    raise ValueError("write_mode must be overwrite or append.")

if parse_mode.upper() not in ["PERMISSIVE", "FAILFAST", "DROPMALFORMED"]:
    raise ValueError("Invalid XML parse mode.")


# ============================================================
# 3. VALIDATE SOURCE FILE
# ============================================================

try:
    dbutils.fs.ls(source_file_path)

    print("Source file validation successful.")

except Exception as e:
    raise FileNotFoundError(
        f"Source file not found: {source_file_path}"
    ) from e


# ============================================================
# 4. CREATE TARGET SCHEMA IF NOT EXISTS
# ============================================================

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{target_schema}`"
)


# ============================================================
# 5. READ XML FILE
# ============================================================

df = (
    spark.read
    .format("xml")
    .option("rowTag", row_tag)
    .option("mode", parse_mode)
    .option("inferSchema", "true")
    .option("samplingRatio", "1.0")
    .load(source_file_path)
)

print("XML file read successfully.")

print("Original XML schema:")
df.printSchema()


# ============================================================
# 6. VALIDATE XML BUSINESS COLUMNS
# ============================================================

if len(df.columns) == 0:
    raise ValueError(
        f"No XML business columns detected. "
        f"Check whether row_tag='{row_tag}' matches the XML structure."
    )

print("Business columns detected:", len(df.columns))

print("Business column names:", df.columns)


# ============================================================
# 7. CLEAN COLUMN NAMES
# ============================================================

def clean_column_name(column_name):

    # Replace spaces and special characters with underscores
    cleaned = re.sub(r"[^a-zA-Z0-9_]", "_", column_name)

    # Remove repeated underscores
    cleaned = re.sub(r"_+", "_", cleaned)

    # Remove leading and trailing underscores
    cleaned = cleaned.strip("_")

    # Convert to lowercase
    cleaned = cleaned.lower()

    return cleaned


# ============================================================
# 8. HANDLE DUPLICATE COLUMN NAMES
# ============================================================

def make_unique_columns(columns):

    result = []
    seen = {}

    for column in columns:

        cleaned = clean_column_name(column)

        if not cleaned:
            cleaned = "column"

        if cleaned in seen:
            seen[cleaned] += 1
            cleaned = f"{cleaned}_{seen[cleaned]}"

        else:
            seen[cleaned] = 0

        result.append(cleaned)

    return result


cleaned_columns = make_unique_columns(df.columns)

# Rename columns
df = df.toDF(*cleaned_columns)

print("Column names cleaned successfully.")


# ============================================================
# 9. ADD AUDIT COLUMNS
# ============================================================

df = (
    df
    .withColumn("load_dt", F.current_timestamp())
    .withColumn("source", F.lit(source_file_path))
)

print("Audit columns added successfully.")


# ============================================================
# 10. VALIDATE FINAL DATAFRAME
# ============================================================

expected_audit_columns = ["load_dt", "source"]

for column in expected_audit_columns:

    if column not in df.columns:
        raise ValueError(
            f"Required audit column missing: {column}"
        )

business_columns = [
    column
    for column in df.columns
    if column not in expected_audit_columns
]

if len(business_columns) == 0:
    raise ValueError(
        "No business columns found. Bronze ingestion stopped."
    )

print("Final business columns:", len(business_columns))
print("Final total columns:", len(df.columns))

print("Final schema:")
df.printSchema()


# ============================================================
# 11. WRITE DATA TO BRONZE DELTA TABLE
# ============================================================

(
    df.write
    .format("delta")
    .mode(write_mode.lower())
    .option("overwriteSchema", "true")
    .saveAsTable(target_table_full_name)
)

print("Bronze table written successfully.")


# ============================================================
# 12. ADD TABLE COMMENT
# ============================================================

spark.sql(
    f"""
    COMMENT ON TABLE {target_table_full_name}
    IS '{table_comment.replace("'", "''")}'
    """
)


# ============================================================
# 13. ADD COLUMN COMMENTS
# ============================================================

spark.sql(
    f"""
    ALTER TABLE {target_table_full_name}
    ALTER COLUMN load_dt
    COMMENT 'Timestamp when the record was loaded into Bronze'
    """
)

spark.sql(
    f"""
    ALTER TABLE {target_table_full_name}
    ALTER COLUMN source
    COMMENT 'Source file path from which the record was ingested'
    """
)


# ============================================================
# 14. FINAL VALIDATION
# ============================================================

bronze_df = spark.table(target_table_full_name)

print("Bronze ingestion completed successfully.")

print("Total records:", bronze_df.count())

print("Total columns:", len(bronze_df.columns))

print("Bronze table schema:")
bronze_df.printSchema()

display(bronze_df.limit(10))